# Ceneo Web Scraper

In [ ]:
#%pip install requests beautifulsoup4

import json
import requests
from bs4 import BeautifulSoup

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Pobranie opinii o produkcie

In [ ]:
product_id = "124893467"
url = f"https://www.ceneo.pl/{product_id}#tab=reviews"

### Wysłanie żądania 

In [ ]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/147.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers, timeout=15)

print("Status:", response.status_code)

response.raise_for_status()

Status: 200


### Wyodrębnienie z kodu strony fragmentów odpowiadających poszczególnym opiniom

In [ ]:
page_dom = BeautifulSoup(response.text, "html.parser")

opinions = page_dom.select(
    "div.js_product-review:not(.user-post--highlight)"
)

print("Znaleziono opinii:", len(opinions))

Znaleziono opinii: 10


### Funkcja zabezpieczająca przed błędem NoneType

In [ ]:
def get_text(parent, selector, default=None):
    element = parent.select_one(selector)
    return element.get_text(" ", strip=True) if element else default

### Wyciągnięcie dla każdek opinii jej składowych

In [ ]:
all_opinions = []
for opinion in opinions:
     single_opinion = {
        "opinion_id" : opinion.get("data-entry-id"),
        "author" : opinion.select_one("span.user-post__author-name").text.strip(),
        "recommendation": opinion.select_one("span.user-post__author-recomendation > em").text.strip() if opinion.select_one("span.user-post__author-recomendation > em") else None,
        "stars" : opinion.select_one("span.user-post__score-count").text.strip(),
        "content" : opinion.select_one("div.user-post__text").text.strip(),
        "pros": [
             p.get_text(" ", strip=True)
             for p in opinion.select("div.review-feature__item--positive")
        ],
        "cons" : [
            c.get_text(" ", strip=True)
            for c in opinion.select("div.review-feature__item--negative")
        ],
        "helpful" : opinion.select_one("button.vote-yes > span").text.strip(),
        "unhelpful" : opinion.select_one("button.vote-no > span").text.strip(),
        "publish_date" : opinion.select_one("span.user-post__published > time:nth-child(1)[datetime]").text.strip(),
        "purchase_date" : opinion.select_one("span.user-post__published > time:nth-child(2)[datetime]").text.strip() if opinion.select_one("span.user-post__published > time:nth-child(2)[datetime]") else None
    }
     all_opinions.append(single_opinion)

print("Zapisano opinii w pamięci:", len(all_opinions))

Zapisano opinii w pamięci: 10


### Zapis do JSON

In [ ]:
with open("opinions.json", "w", encoding="UTF-8") as json_file:
    json.dump(all_opinions, json_file, indent=4, ensure_ascii=False)